# Lab 1 — Version 3

## Approach
This is a small refinement of Version 2 using the same author/context features,
TF-IDF/SVD and cached multilingual embeddings. It adds no encoder training.

Two gradient boosting settings were compared with the original Version 2:
- **smooth:** at most 15 leaves, L2 regularization 3, learning rate 0.05, up to 600 iterations.
- **richer:** at most 63 leaves, at least 30 samples per leaf, L2 regularization 5,
  learning rate 0.05, up to 600 iterations.

For each setting the SVD/tabular and embeddings/tabular predictions are averaged,
then mixed with 15% TF-IDF logistic-regression predictions, as in Version 2.
Equal blends of the original and each new setting were also tested.
The decision multipliers for below and typical were varied over {0.9, 1.0, 1.1}
and {1.4, 1.6, 1.8}, with above fixed at 1.0.

## Validation
The same three author-disjoint training/validation partitions were used (seeds
42, 123 and 2026). Every vocabulary, SVD/PCA transform and classifier was fitted
only on fitting authors. All settings were selected by mean validation macro-F1;
no test labels entered this selection. There were 45 combinations per split.

Before test evaluation, the candidate was required to improve all three validation
scores and increase their mean by at least 0.002 over the fixed Version 2 rule.

| Candidate family, best shared decision rule | Mean validation macro-F1 |
|---|---:|
| v2 | 0.6513 |
| smooth | 0.6531 |
| richer | 0.6543 |
| blend_smooth | 0.6527 |
| blend_richer | 0.6529 |

The fixed submitted Version 2 score is 0.6506.
Selected: **richer**, below multiplier **1.1**,
typical multiplier **1.6**, mean score **0.6543**.
These multipliers adjust class decisions, not calibrated probabilities.

## Test results
| Model | Macro-F1 | QWK | Accuracy |
|---|---:|---:|---:|
| Official baseline | 0.5410 | 0.4576 | 0.5465 |
| Version 1 | 0.6052 | 0.5240 | 0.6444 |
| Version 2 | 0.6412 | 0.5699 | 0.6860 |
| Version 3 | **0.6436** | **0.5720** | **0.6865** |

Per-class F1: below 0.5246, typical 0.7486,
above 0.6576. The official evaluate.py checks all 17,623 predictions.

## Limitations
This is another evaluation on the same test set, which was already inspected in
earlier versions. Configuration selection uses training-only validation, but this
is not a fresh independent test or a statistical significance claim. Repeated
validation partitions overlap. The snapshot and 128-token encoder limitations
of Version 2 still apply.

## Reproduction
Version 3 requires the original project dependencies and the prepared Version 2
cache in results/v2_cache/. On a fresh setup, follow the data and embedding
preparation steps in lab1_v2.ipynb first.

```bash
python improved_v3.py --stage validate
python improved_v3.py --stage final
python evaluate.py --predictions lab1_submission_v3/predictions.csv
```

To repeat final training with the saved selection and existing cache, only the last
two commands are needed. Code, metrics and predictions are submitted; intermediate
data and embeddings remain in the Git-ignored results/ directory.
The cell below displays saved results without retraining.


In [1]:
import json
from pathlib import Path
results = json.loads(Path("lab1_submission_v3/metrics.json").read_text())
print(json.dumps(results["test"], indent=2))


{
  "macro_f1": 0.6436,
  "qwk": 0.572,
  "accuracy": 0.6865,
  "f1_by_class": [
    0.5245994612221749,
    0.7485929578119913,
    0.6576038243202869
  ],
  "confusion_matrix": [
    [
      1850,
      1653,
      38
    ],
    [
      1580,
      8047,
      864
    ],
    [
      82,
      1308,
      2201
    ]
  ]
}
